# Intent Classification on CLINC150 with Deep Neural Networks

**Deep Neural Networks - course project.**

**Goal:** build a proof-of-concept intent classifier with DNNs and evaluate whether it could
replace the LLM-based intent classifier used in my master's thesis project **SettleIn**
(an Agentic AI system for immigrant assistance in Serbia,
[github.com/IlorDash/settle-in](https://github.com/IlorDash/settle-in)).

Today SettleIn classifies user intent with a GPT-4o-mini API call. This project asks: can a small,
**fast, deterministic, cost-free** DNN do the job instead - including detecting **out-of-scope**
questions (queries outside supported topics)?

**Dataset - CLINC150:** 150 in-scope intent classes across 10 domains (banking, travel, work, ...),
plus a dedicated **out-of-scope (OOS)** class. We use `data_full.json`: 100 train / 20 val / 30 test
samples per in-scope intent, and 100 / 100 / 1000 OOS samples.

> **Application to SettleIn (no retraining):** once the classifier is trained, we select the CLINC150
> intents that match topics SettleIn supports (e.g. banking, travel, work, utilities) and treat every
> other intent - plus the native OOS class - as out-of-scope. The *same* model then acts as SettleIn's
> gatekeeper: route a supported topic, reject an unsupported one. That is exactly the decision
> SettleIn's orchestrator makes today with a GPT-4o-mini call. (A production system would later refine
> the relevant-intent set with real query logs - standard future work.)

## Plan
1. Load the data
2. Explore the data (EDA)
3. Turn text into numbers (TF-IDF vs. learned embeddings)
4. Baseline: TF-IDF + Dense MLP
5. Deep model 1: Embedding + pooling
6. Deep model 2: Embedding + Conv1D / BiLSTM
7. Tuning & comparison
8. Out-of-scope detection (confidence threshold vs. OOS-as-class)
9. Application to SettleIn: relevant-intent subset as a gatekeeper (no retraining)

## 1. Load the dataset

`data_full.json` has six top-level keys. Each maps to a list of `[text, intent]` pairs:

| Key | Contents | Size |
|-----|----------|------|
| `train` / `val` / `test` | in-scope, 150 intents | 15000 / 3000 / 4500 |
| `oos_train` / `oos_val` / `oos_test` | out-of-scope (label `"oos"`) | 100 / 100 / 1000 |

We keep the in-scope and out-of-scope splits **separate** because out-of-scope is open-ended
("anything the system doesn't support") and cannot be fully enumerated as training data. Keeping
them apart lets us test two OOS strategies later: detect OOS from the model's *confidence* (trained
on in-scope only), or add OOS as an explicit 151st class - and compare the two.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
DATA_PATH = Path("data/data_full.json")

with open(DATA_PATH, encoding="utf-8") as f:
    clinc = json.load(f)

list(clinc.keys())

In [ ]:
def split_to_frame(pairs: list) -> pd.DataFrame:
    """Convert a list of [text, intent] pairs into a tidy DataFrame."""
    return pd.DataFrame(pairs, columns=["text", "intent"])


train_df = split_to_frame(clinc["train"])
val_df = split_to_frame(clinc["val"])
test_df = split_to_frame(clinc["test"])

oos_train_df = split_to_frame(clinc["oos_train"])
oos_val_df = split_to_frame(clinc["oos_val"])
oos_test_df = split_to_frame(clinc["oos_test"])

In [ ]:
train_df.head()

### Why three sets: train / validation / test

CLINC150 ships with **three** splits rather than the usual two. This structure exists to keep the
final score **honest**, and it shapes how we use each one.

| Split | Who uses it | Purpose | Touched |
|-------|-------------|---------|---------|
| **train** | the model | learns its weights (`fit`) | every epoch |
| **validation** | us | tuning decisions: when to stop, which model, which settings | constantly while developing |
| **test** | us, once | final unbiased score on unseen data | only at the very end |

**The trap with two sets:** every time we look at a score and *change something* (stop early, pick
64 units over 32, choose one model over another), that data has influenced our choices. If those
choices are guided by the test set, the test score becomes optimistic - it no longer predicts
performance on truly new data. The validation set is the one we are allowed to tune against; the
test set stays locked away until the final measurement.

So the three roles are: train to learn, validation to tune, and test to judge exactly once - an
approach called *hold-out evaluation*. Because the dataset provides the splits ready-made, every
researcher reports results on the same untouched test set, which keeps published numbers comparable.

## 2. Explore the data (EDA)

Before modelling we measure properties of the raw text, because each one feeds a concrete decision
later:

- **Class balance** -> is accuracy a fair metric, or do we lean on macro-F1?
- **Query length** -> what sequence length should we pad/truncate to?
- **Vocabulary size** -> how many tokens should the vectorizer keep?
- **Example queries** -> what does the text actually look like?
- **In-scope vs out-of-scope** -> can OOS be told apart by surface features, or only by meaning?

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

### Class balance

Each in-scope intent has the same number of training samples, so the dataset is balanced. That
means plain accuracy is not misleading - but with 150 classes we still report **macro-F1**, which
averages performance across classes equally and exposes any single intent the model handles badly.

In [ ]:
intent_counts = train_df["intent"].value_counts()
intent_counts.describe()

### The 150 intents

The full list of intent labels. We use this later to pick the subset relevant to SettleIn
(banking, travel, work, utilities, ...) and treat the rest as out-of-scope.

In [ ]:
sorted(train_df["intent"].unique())

### Query length

How long are the queries, in words? This sets the sequence length we pad/truncate to for the
embedding models: too short loses information, too long wastes computation on padding.

In [ ]:
train_df["n_words"] = train_df["text"].str.split().str.len()
train_df["n_words"].describe()

In [ ]:
# Percentiles guide the padding length: cover almost every query without an extreme max.
train_df["n_words"].quantile([0.5, 0.95, 0.99, 1.0])

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(train_df["n_words"], bins=range(0, train_df["n_words"].max() + 2))
plt.xlabel("words per query")
plt.ylabel("number of queries")
plt.title("Query length distribution (in-scope train)")
plt.show()

### Vocabulary size

The number of distinct words across the training set. This sets a ceiling for the vectorizer's
vocabulary - the number of TF-IDF features, or the Embedding layer's input dimension. The TF-IDF
vectorizer in Section 3.2 keeps fewer than this (about 2770), because `min_df=2` removes the many
words that appear in only a single query.

In [ ]:
vocabulary = set()
for text in train_df["text"]:
    vocabulary.update(text.lower().split())

len(vocabulary)

### Example queries per intent

What does the raw text look like? Three examples from the first five intents (alphabetically).

In [ ]:
for intent in sorted(train_df["intent"].unique())[:5]:
    print(f"\n=== {intent} ===")
    for text in train_df.loc[train_df["intent"] == intent, "text"].head(3):
        print(" -", text)

### In-scope vs out-of-scope

First some real out-of-scope queries, then a comparison of length distributions. If in-scope and
OOS look similar on the surface, OOS detection must rely on **meaning**, not superficial cues - the
motivation for the confidence-threshold approach.

In [ ]:
oos_test_df["text"].head(10).tolist()

In [ ]:
oos_test_df["n_words"] = oos_test_df["text"].str.split().str.len()

plt.figure(figsize=(8, 4))
sns.histplot(train_df["n_words"], bins=range(0, 30), stat="density",
             label="in-scope", color="C0", alpha=0.6)
sns.histplot(oos_test_df["n_words"], bins=range(0, 30), stat="density",
             label="out-of-scope", color="C1", alpha=0.6)
plt.xlabel("words per query")
plt.ylabel("density")
plt.title("Query length: in-scope vs out-of-scope")
plt.legend()
plt.show()

## 3. Turn text into numbers

A network can't read strings. We need two separate conversions:

1. **Labels**: intent *names* -> integer class IDs (this section, 3.1).
2. **Features**: query *text* -> numeric vectors (3.2, TF-IDF for the baseline).

We start with the labels because every model - baseline and deep - needs them in this form. Using
`sparse_categorical_crossentropy` later means the labels stay as plain integers, with no one-hot
encoding step required.

### 3.1 Encode the labels

`LabelEncoder` maps each of the 150 intent names to an integer 0-149. We **fit on the training
intents only**, then reuse that mapping for val and test so a given intent always gets the same ID.
We keep the encoder to translate predictions back to names later via `inverse_transform`.

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
label_encoder.fit(train_df["intent"])

y_train = label_encoder.transform(train_df["intent"])
y_val = label_encoder.transform(val_df["intent"])
y_test = label_encoder.transform(test_df["intent"])

num_classes = len(label_encoder.classes_)
num_classes

In [ ]:
# class i corresponds to label_encoder.classes_[i]
list(enumerate(label_encoder.classes_[:5]))

### 3.2 Vectorize the text (TF-IDF)

`TfidfVectorizer` turns each query into a numeric vector - one weight per word in the *vocabulary*.
The vocabulary is the set of distinct words the vectorizer knows about, collected from the training
text. We **fit on the training text only** (the same anti-leakage rule as scaling in the course
notebooks), then transform every split with that fixed vocabulary.

`min_df=2` drops words appearing in only a single query: they can't help the model generalize and
would only inflate the feature space. The sections below unpack what TF-IDF computes, what `fit` and
`transform` do, and what the resulting matrix actually looks like.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(min_df=2)
X_train = vectorizer.fit_transform(train_df["text"])
X_val = vectorizer.transform(val_df["text"])
X_test = vectorizer.transform(test_df["text"])

X_train.shape, X_val.shape, X_test.shape

**Why 2770 features and not the 5863 from the EDA?** The earlier count split on spaces and kept
*every* distinct token, including the many words that appear in just one query. `TfidfVectorizer` is
stricter: `min_df=2` drops every word occurring in fewer than two queries (a large share of those
5863), and its tokenizer also discards one-letter tokens like "i" and "a". What remains - 2770
words - are the ones common enough to be useful, shared features.

### The math behind TF-IDF

TF-IDF converts a query into numbers with **no training** - it is pure counting plus one formula.
Each word's weight is built from two factors:

**TF (term frequency)** - how often a word appears *in this one query*.
- More occurrences in the query -> higher weight.
- It is a property of a (word, query) pair, so it is recomputed for every query during `transform`.

**IDF (inverse document frequency)** - how rare a word is *across all training queries*.
- A single fixed number per word, learned during `fit` and stored in `vectorizer.idf_`.
- scikit-learn's formula, where `N` = number of training queries and `df` (the *document
  frequency*) = how many of them contain the word:

```
idf(word) = ln( (1 + N) / (1 + df(word)) ) + 1
```

- A word in *many* queries ("to", "what") -> small idf. A *rare* word ("italian") -> large idf.

**The stored weight** is their product, after which each row is scaled to unit length:

```
weight(word, query) = tf(word, query) * idf(word)
```

Putting it together: TF measures local frequency (within one query) and IDF measures global rarity
(across the whole corpus), so a high weight marks a word that is frequent here but rare overall -
exactly the kind of word that signals intent. Common filler words such as "the" or "to" sink toward
zero automatically, with no need for a stop-word list.

In [ ]:
# Each word's learned IDF, lowest (most common) first.
# idf_ is aligned with get_feature_names_out() (same column order), not with vocabulary_.items().
pd.Series(vectorizer.idf_, index=vectorizer.get_feature_names_out()).sort_values().head(10)

In [ ]:
# The non-zero TF-IDF weights of the first training query, highest first.
first_query = pd.Series(
    X_train[0].toarray().ravel(),
    index=vectorizer.get_feature_names_out(),
)
first_query[first_query > 0].sort_values(ascending=False)

### fit, transform, fit_transform - and why only train is fitted

The vectorizer is a **stateful object** - a translator that first needs an internal dictionary
(which words exist, their column, their idf) before it can convert anything.

| Method | What it does | Changes the object? | Returns |
|--------|--------------|---------------------|---------|
| `fit(texts)` | reads texts and **builds** the vocabulary + idf inside the object | yes | the object |
| `transform(texts)` | converts texts into vectors **using the stored dictionary** | no (read-only) | the matrix |
| `fit_transform(texts)` | `fit` then `transform`, in one call | yes | the matrix |

After `fit`, the learned state lives in attributes ending with `_`: `vectorizer.vocabulary_`
(word -> column index) and `vectorizer.idf_` (one number per word). The trailing underscore is
scikit-learn's convention for "learned from data".

We call `fit_transform` on train but only `transform` on val and test, for two reasons:

1. **No data leakage.** *Data leakage* is when information from the validation or test set influences
   training or preprocessing, making the final score look better than it truly is. Because `fit`
   measures statistics from whatever it sees, it must touch the training data only.
2. **Matching columns.** The model expects a fixed 2770-column input. `transform` forces every split
   into that same vocabulary; a val/test word unseen in training is simply dropped (no column),
   exactly as a brand-new user query would be in production.

This mirrors the course notebooks' pattern: `MinMaxScaler.fit(train)` then `transform(test)` - learn
the rule on the training data, then apply it everywhere.

### What we get back: a sparse matrix

`transform` returns a matrix with **one row per query** and **one column per vocabulary word** (2770
columns). Cell `[query, word]` holds that word's TF-IDF weight in that query, or `0` if the word is
absent. Each row is therefore a *bag-of-words* representation: it records which words appear and how
strongly, but discards their order.

Because a query uses only ~8 words out of 2770, **almost every cell is 0**. A matrix that is mostly
zeros is called a *sparse matrix*. Instead of storing 15000 x 2770 = ~41 million numbers, SciPy keeps
*only the non-zero entries and their positions* - that is why `X_train` prints as a one-line summary,
not a grid, and why it stays small in memory.

The toy example below makes it concrete: 3 sentences become a small grid where each row has a few
weights and the rest are zeros. The real `X_train` is the same picture at 15000 x 2770.

In [ ]:
# A toy TF-IDF on 3 short sentences, shown as a full (dense) grid so the zeros are visible.
# The real X_train is this same idea, just 15000 x 2770.
demo_texts = [
    "transfer money to my account",
    "what is my account balance",
    "book a flight to rome",
]
demo_vectorizer = TfidfVectorizer()
demo_matrix = demo_vectorizer.fit_transform(demo_texts)

pd.DataFrame(
    demo_matrix.toarray().round(2),
    columns=demo_vectorizer.get_feature_names_out(),
    index=demo_texts,
)

In [ ]:
# How sparse is the real matrix? Non-zero cells vs total cells.
non_zero = X_train.nnz
total = X_train.shape[0] * X_train.shape[1]
print(f"non-zero cells: {non_zero:,} out of {total:,}")
print(f"fraction non-zero: {non_zero / total:.4%}")

## 4. Baseline model: TF-IDF + Dense MLP

A *baseline* is a deliberately simple model whose job is to set a score the deeper models must beat:
if a more complex architecture cannot outperform a plain one, the complexity is not worth it.

This baseline is a small feed-forward network (a *multi-layer perceptron*, or MLP) that takes the
2770 TF-IDF features of a query and predicts one of the 150 intents. It is the same kind of network
as the classification notebook from the course, with two changes for multi-class output:

- the output layer has **150 units with a softmax** activation (one probability per intent) instead
  of a single sigmoid;
- the loss is **`sparse_categorical_crossentropy`**, which reads the integer labels directly.

*Softmax* is the activation on that output layer. It takes the 150 raw scores the network produces
(any numbers, positive or negative) and turns them into 150 probabilities between 0 and 1 that add
up to 1 - a probability distribution over the intents. It works by exponentiating each score and
dividing by the sum of all of them, so a larger score becomes a larger probability while the total
stays at 1. The predicted intent is the one with the highest probability, and that probability also
measures how confident the model is - exactly the signal we will later use to flag out-of-scope
queries. (Softmax is the multi-class version of the sigmoid used for binary classification.)

#### Softmax in one picture

The left chart shows five example raw scores (logits) - they can be any size, even negative. The
right chart shows the same five values after softmax: each is now between 0 and 1, and together they
add up to 1, with the largest score keeping the largest share. Softmax does this by raising `e` to
each score (which is always positive) and then dividing by the total. Try editing `example_logits`
to see how the probabilities shift.

In [ ]:
# Softmax turns raw scores (logits) into probabilities that sum to 1.
example_logits = np.array([4.0, 2.0, 1.0, 0.5, -1.0])
probabilities = np.exp(example_logits) / np.exp(example_logits).sum()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].bar(range(len(example_logits)), example_logits, color="gray")
axes[0].set_title("Raw scores (logits)")
axes[0].set_xlabel("class")

axes[1].bar(range(len(probabilities)), probabilities, color="C0")
axes[1].set_title(f"After softmax (bars sum to {probabilities.sum():.2f})")
axes[1].set_xlabel("class")
axes[1].set_ylabel("probability")
plt.show()

In [ ]:
# Keras Dense layers need a dense array, so expand the sparse matrices to float32.
# 15000 x 2770 floats is about 166 MB - safe in memory.
X_train_dense = X_train.toarray().astype("float32")
X_val_dense = X_val.toarray().astype("float32")
X_test_dense = X_test.toarray().astype("float32")

X_train_dense.shape

### Build the network

One hidden layer is enough for a baseline. `Dense(256)` with a *ReLU* activation learns combinations
of the TF-IDF features; a `Dropout(0.5)` layer randomly zeroes half of those activations during
training to reduce overfitting; and the final `Dense(150)` with softmax produces the per-intent
probabilities. We fix a random seed so the run is reproducible.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

tf.keras.utils.set_random_seed(42)

n_features = X_train_dense.shape[1]

baseline_model = Sequential([
    Input(shape=(n_features,)),
    Dense(256, activation="relu"),
    Dropout(0.5),
    Dense(num_classes, activation="softmax"),
])

baseline_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

baseline_model.summary()

### Train with early stopping

We train on the TF-IDF features and watch the **validation loss**. *Early stopping* halts training
once `val_loss` stops improving for `patience` epochs, and `restore_best_weights=True` rolls the
model back to its best epoch - so we neither under- nor over-train. The test set is not touched
here; it stays reserved for the final comparison.

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
)

history = baseline_model.fit(
    X_train_dense, y_train,
    validation_data=(X_val_dense, y_val),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop],
)

In [ ]:
history_df = pd.DataFrame(history.history)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
history_df[["loss", "val_loss"]].plot(ax=axes[0], title="Loss")
history_df[["accuracy", "val_accuracy"]].plot(ax=axes[1], title="Accuracy")
plt.show()

### Reading the training curves

Both plots share the same x-axis: the **epoch**, one full pass of training over all 15000 training
queries.

- **Loss** (left) is the quantity the model minimizes; lower is better. The `loss` line is measured
  on the training set, `val_loss` on the validation set.
- **Accuracy** (right) is the fraction of correct predictions; higher is better, again split into
  training and validation.

How to read them:
- Both losses falling and then flattening means the model learned and converged.
- If training keeps improving while validation stalls or worsens, that gap is **overfitting** - the
  model is memorizing the training set instead of generalizing. `val_loss` turning upward is the
  classic sign, and it is exactly where early stopping halts and restores the best epoch.
- The epoch with the lowest `val_loss` is the model we keep.

### Evaluate on the validation set

We report two numbers on the validation set. *Accuracy* is the fraction of queries given the correct
intent. *Macro-F1* is the F1 score (the balance of precision and recall) averaged equally over all
150 intents, so a handful of badly handled intents cannot hide behind the rest. These are the
baseline numbers the deeper models will try to beat.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

val_pred = baseline_model.predict(X_val_dense).argmax(axis=1)

print(f"validation accuracy: {accuracy_score(y_val, val_pred):.4f}")
print(f"validation macro-F1: {f1_score(y_val, val_pred, average='macro'):.4f}")

## 5. Deep model 1: Embedding + pooling

The baseline treated each word as an isolated feature: "transfer" and "transaction" were two
unrelated columns, and word order was thrown away. This model fixes the first problem with **word
embeddings**.

An *embedding* is a short list of numbers (a vector) attached to each word and *learned during
training*. Words used in similar ways end up with similar vectors, so the model can generalize -
seeing "transfer" at training time also teaches it something about "transaction". Instead of one
fixed bag-of-words vector per query, each query becomes a *sequence* of these word vectors.

The plan for this section:
1. turn each query into a padded sequence of integer word IDs;
2. let an `Embedding` layer map each ID to a learned vector;
3. average those vectors into one summary vector (pooling) and classify it with a softmax.

### 5.1 Turn text into padded integer sequences

The embedding layer needs each word as an integer ID, not a TF-IDF weight. Keras' `TextVectorization`
layer builds a vocabulary from the training text and maps each query to a list of integer IDs.

Two settings matter:
- `max_tokens` caps the vocabulary at the most frequent words (we use 5000).
- `output_sequence_length` fixes every sequence to the same length by *padding* short queries with
  zeros and truncating long ones. From the EDA, 99% of queries are <= 17 words, so a length of 20
  covers almost all of them with little waste.

We `adapt` the layer on the training text only (the same fit-on-train rule), then apply it to every
split.

In [ ]:
from tensorflow.keras.layers import TextVectorization

MAX_LEN = 20
VOCAB_SIZE = 5000

vectorize_layer = TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=MAX_LEN,
)
vectorize_layer.adapt(train_df["text"].to_numpy())

X_train_seq = vectorize_layer(train_df["text"].to_numpy()).numpy()
X_val_seq = vectorize_layer(val_df["text"].to_numpy()).numpy()
X_test_seq = vectorize_layer(test_df["text"].to_numpy()).numpy()

X_train_seq.shape

In [ ]:
# One query as raw text vs. its padded integer sequence (0 = padding).
print(train_df["text"].iloc[0])
print(X_train_seq[0])

### 5.2 Build the embedding model

Three new layers replace the TF-IDF input:

- **`Embedding`** holds one learned vector per vocabulary word (here 64 numbers each). It turns a
  length-20 sequence of IDs into a 20 x 64 grid of word vectors. `mask_zero=True` tells the later
  layers to ignore the padding positions.
- **`GlobalAveragePooling1D`** averages the 20 word vectors into a single 64-number vector that
  summarizes the whole query.
- A final **`Dense(150)` softmax** classifies that summary, exactly as in the baseline.

Averaging still ignores word order - we address that in Section 6 - but the word vectors themselves
are learned and shared, which is the gain over bag-of-words.

In [ ]:
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D

EMBED_DIM = 64
vocab_size = vectorize_layer.vocabulary_size()

tf.keras.utils.set_random_seed(42)

embedding_model = Sequential([
    Input(shape=(MAX_LEN,), dtype="int32"),
    Embedding(input_dim=vocab_size, output_dim=EMBED_DIM, mask_zero=True),
    GlobalAveragePooling1D(),
    Dropout(0.5),
    Dense(num_classes, activation="softmax"),
])

embedding_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

embedding_model.summary()

### 5.3 Train and evaluate

We reuse the baseline's recipe: train on the integer sequences with early stopping on the validation
loss, then read the loss and accuracy curves the same way. The validation accuracy and macro-F1 at
the end are what we compare against the baseline (0.915 accuracy).

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
)

embedding_history = embedding_model.fit(
    X_train_seq, y_train,
    validation_data=(X_val_seq, y_val),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop],
)

In [ ]:
embedding_history_df = pd.DataFrame(embedding_history.history)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
embedding_history_df[["loss", "val_loss"]].plot(ax=axes[0], title="Loss")
embedding_history_df[["accuracy", "val_accuracy"]].plot(ax=axes[1], title="Accuracy")
plt.show()

In [ ]:
emb_val_pred = embedding_model.predict(X_val_seq).argmax(axis=1)

print(f"validation accuracy: {accuracy_score(y_val, emb_val_pred):.4f}")
print(f"validation macro-F1: {f1_score(y_val, emb_val_pred, average='macro'):.4f}")

## 6. Deep model 2: Embedding + Conv1D

Average pooling in Section 5 ignored word order. A **1D convolution** brings it back.

**What a 1D convolution does.** A *filter* is a small window (here 5 words wide) that slides along the sequence of word vectors, one step at a time. At each position it looks at those 5 consecutive words and outputs a single number: how strongly that window matches the pattern the filter has learned. Sliding it across the whole query gives a row of such numbers - high where the pattern appears, low elsewhere.

```
query:   book   a   flight   to   rome
         [-- filter --]                  -> 0.1
                [-- filter --]           -> 0.9   <- matches "a flight to rome"
                       [-- filter --]    -> 0.2
```

- Each filter learns one local word-order pattern (an *n-gram* feature) such as "book a flight" or "how do i". We use 128 filters, so the layer learns 128 patterns in parallel.
- Because the same filter slides everywhere, a pattern is detected no matter *where* in the query it appears (position independence).
- **`GlobalMaxPooling1D`** then takes, for each filter, the single highest value along the query - "did this pattern show up anywhere, and how strongly?". Those 128 maxima form the query summary, which a softmax classifies.

This is the key difference from Sections 4-5: TF-IDF and average pooling treat words as an unordered bag, while the convolution reacts to short *sequences* of words. We reuse the same padded integer sequences (`X_train_seq`) and vocabulary from Section 5; only the model changes.

In [ ]:
from tensorflow.keras.layers import Conv1D, GlobalMaxPooling1D

tf.keras.utils.set_random_seed(42)

conv_model = Sequential([
    Input(shape=(MAX_LEN,), dtype="int32"),
    Embedding(input_dim=vocab_size, output_dim=EMBED_DIM),
    Conv1D(128, kernel_size=5, activation="relu"),
    GlobalMaxPooling1D(),
    Dropout(0.5),
    Dense(num_classes, activation="softmax"),
])

conv_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

conv_model.summary()

In [ ]:
early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

conv_history = conv_model.fit(
    X_train_seq, y_train,
    validation_data=(X_val_seq, y_val),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop],
)

In [ ]:
conv_history_df = pd.DataFrame(conv_history.history)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
conv_history_df[["loss", "val_loss"]].plot(ax=axes[0], title="Loss")
conv_history_df[["accuracy", "val_accuracy"]].plot(ax=axes[1], title="Accuracy")
plt.show()

In [ ]:
conv_val_pred = conv_model.predict(X_val_seq).argmax(axis=1)

print(f"validation accuracy: {accuracy_score(y_val, conv_val_pred):.4f}")
print(f"validation macro-F1: {f1_score(y_val, conv_val_pred, average='macro'):.4f}")